# Tutorial to run a dynamical simulation

To run only the dynamical evolution you can run the following script:
```
python pypopsyn/simulator/simulate_population_dyn.py --save_dir output/sim_dyn
```

To change the initial parameters, the user can directly modify the simulator configuration in `pypopsyn/simulator/config_simulator.py` or alternatively parsing a JSON file containing custom parameters for the simulation.

This will create a population of neutron stars according to the initial conditions specified in the `pypopsyn/simulator/config_simulator.py` and evolve it in time dynamically.
The output is saved in the specified output folder and it consists of the following files:
* `final_pop_dyn.csv` containing the information on the final positions and velocities of neutron stars in the Galaxy.
* `.json` and `.log` files containing the timing profiles for the simulation, if enabled.
* `configuration.json` file containing the configuration parameters for reproducibility.

You can look at the paper [Ronchi et al. 2022](https://ui.adsabs.harvard.edu/abs/2021ApJ...916..100R/abstract) for an application of this simulator and for more details on the assumed physical models.

In [ ]:
import argparse
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys

import utilities.plot_settings
from pypopsyn.simulator.config_simulator import cfg
import pypopsyn.simulator.stellar_dynamics.coordinate_conversions as cc
from pypopsyn.simulator.simulate_population_dyn import simulate_population

## Setup and run the simulation

Change some parameters in the imported configuration file.
For example here we can change the following:
1. `NS_number`, number of neutron stars to simulate.
2. `t_age_max`, the maximum age for the simulated neutron stars in [yr].
3. `kick_model`, the kick-velocity model, you can choose either a Maxwell distribution, `km_maxwell` or an exponential distribution, `km_exp`.
4. `sigma_k`, dispersion of the kick-velocity distribution if `kick_model = km_maxwell`.
5. `vk_c`, characteristic velocity of the kick-velocity distribution if `kick_model = km_exp`.
6. `h_c`, characteristic scale height of the Galactic exponential disk model.

NOTE: to have a realistic birth rate ($\sim 1$ neutron star per century) you should set the `NS_number` and the `t_age_max` accordingly. However by increasing both `NS_number` and the `t_age_max` the simulation would take more time to run.

In [ ]:
cfg["NS_number"] = 10000
cfg["t_age_max"] = 3.e7
cfg["kick_model"] = "km_maxwell"
cfg["sigma_k"]= 265.0
cfg["vk_c"]= 180.0
cfg["h_c"] = 0.18

Alternatively you can change the parameters in the `parameter_override.json` file in the `tutorials/tutorial_notebooks` folder and pass it to the simulator.

In [ ]:
override_dir = "parameter_override.json"

Specify the output directory where the simulation results will be saved.

In [ ]:
output_dir = "output/sim_dyn"

Run the simulation.

In [ ]:
simulation_args = argparse.Namespace(
    save_dir = output_dir,
    parameter_override = None
)
simulate_population(simulation_args)

## Read the simulation results

In [ ]:
data_dyn = pd.read_csv(
    pathlib.Path().joinpath(output_dir, "final_pop_dyn.csv"),
    delimiter=",",
    header=[0, 1],
)
data_dyn.columns

In [ ]:
r = data_dyn["r"]["[kpc]"].to_numpy()
phi = data_dyn["phi"]["[rad]"].to_numpy()
z = data_dyn["z"]["[kpc]"].to_numpy()
v_r = data_dyn["v_r"]["[km/s]"].to_numpy()
v_phi = data_dyn["v_phi"]["[km/s]"].to_numpy()
v_z = data_dyn["v_z"]["[km/s]"].to_numpy()
age = data_dyn["age"]["[yr]"].to_numpy()

x = r * np.cos(phi)
y = r * np.sin(phi)

## Plot the simulation results

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(
    x,
    y,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=1,
    alpha=0.5,
    rasterized=True,
)
ax.plot(0.0, 8.3, marker="*", color="tab:orange", markersize=20)
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)
plt.xlabel(r"$x$ [kpc]")
plt.ylabel(r"$y$ [kpc]")
plt.legend(bbox_to_anchor=(1, 1), frameon=False, loc=0, fontsize=20)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(
    x,
    z,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=1,
    alpha=0.5,
    rasterized=True,
)

ax.plot(0.0, 0.02, marker="*", color="tab:orange", markersize=20)
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)
plt.xlabel(r"$x$ [kpc]")
plt.ylabel(r"$z$ [kpc]")
plt.legend(bbox_to_anchor=(1, 1), frameon=False, loc=0, fontsize=20)

plt.show()